In [1]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from moabb.paradigms import MotorImagery

from src.data_proc import Our5Class
from src.loaders import *
from src.models import *
from src.train_utils import ClassicTrainer

In [2]:
# Import config
from configs import simple_train_cfg
cfg = simple_train_cfg.get_config()

In [3]:
# # Update config if changed during jupyter runtime
# import importlib
# from configs import simple_train_cfg
# importlib.reload(simple_train_cfg)
# cfg = simple_train_cfg.get_config()

In [4]:
# Set seed
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    # Make deterministic for real research
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_pth = Path("checkpoints") / f"{cfg.model.checkpoint_name}.pth"
# log_pth = Path("results") / f"{cfg.model.checkpoint_name}"

In [5]:
moabb_train_dataset = Our5Class(data_path="our_data/our_5_class", get_last_trials_xor=False)
moabb_val_dataset = Our5Class(data_path="our_data/our_5_class", get_last_trials_xor=True)

paradigm = MotorImagery(
    events=["left_hand", "right_hand", "left_leg", "right_leg", "tongue"],
    n_classes=cfg.eeg.num_classes,
    fmin=cfg.preprocessing.freq_fork[0], fmax=cfg.preprocessing.freq_fork[1],
    resample=cfg.preprocessing.resample_rate,
    tmin=cfg.preprocessing.t_fork[0], tmax=cfg.preprocessing.t_fork[1]
)

# Wrap into torch classes
dataset_config = STFTDatasetConfig(
    paradigm=paradigm,
    window_sec=cfg.preprocessing.window_sec,
    window_overlap=cfg.preprocessing.window_overlap,
    stft_nperseg=cfg.preprocessing.stft_nperseg,
    stft_overlap=cfg.preprocessing.stft_overlap,
    use_cache=cfg.preprocessing.use_cache,
    cache_path=cfg.preprocessing.cache_path,
    dtype=np.float32
)
train_config = copy.deepcopy(dataset_config)
val_config = copy.deepcopy(dataset_config)
train_config.cache_path = Path(cfg.preprocessing.cache_path) / "train"
val_config.cache_path = Path(cfg.preprocessing.cache_path) / "val"

torch_train_dataset = STFTDataset(moabb_train_dataset, train_config)
torch_val_dataset = STFTDataset(moabb_val_dataset, val_config)

In [6]:
# Regularization
train_transform = Compose([
    ClipOutliers(sigma=cfg.regularization.clip_sigma),
    ZScoreNormalize(),
    GaussianNoise(std=cfg.regularization.gaussian_std),
    RandomScale(scale_fork=cfg.regularization.scale_fork),
    TimeShift(max_shift=cfg.regularization.max_shift),
    ChannelDropout(p=cfg.regularization.channel_dropout),
])
# No augmentation for val/test
val_transform = Compose([
    ClipOutliers(sigma=cfg.regularization.clip_sigma),
    ZScoreNormalize(),
])

In [7]:
def init_model(init_scheduler=True):
    model = CNN_BiLSTM(
        in_channels=cfg.eeg.in_channels,
        freq_bins=cfg.preprocessing.stft_nperseg // 2 + 1,  # this is the shape scipy's stft spit's out
        num_classes=cfg.eeg.num_classes,
        hidden_dim=cfg.model.cnnbilstm.hidden_dim,
        rnn_layers=cfg.model.cnnbilstm.rnn_layers,
        dropout=cfg.model.cnnbilstm.dropout
    ).to(device, dtype=torch.float32)

    criterion = nn.CrossEntropyLoss()  # TODO: from config
    optimizer = optim.AdamW(
        model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay
    )  # TODO: from config
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.training.lr_patience, factor=cfg.training.lr_factor
    ) if init_scheduler is True else None
    return model, criterion, optimizer, scheduler

In [8]:
# Init model
model, criterion, optimizer, scheduler = init_model(init_scheduler=True)
# Init trainer
trainer = ClassicTrainer(model, criterion, optimizer, train_transform, val_transform, scheduler, device, cfg)
# Run training
model, criterion, best_history, best_val_loss = trainer.train_loop(
    train_set=torch_train_dataset, val_set=torch_val_dataset, lr_schedule=None
)


Epoch 0/1000:
Train loss: 1.6021, Train acc: 0.2227
Val loss: 1.5992, Val acc: 0.2309

Epoch 1/1000:
Train loss: 1.5701, Train acc: 0.2638
Val loss: 1.6096, Val acc: 0.2498

Epoch 2/1000:
Train loss: 1.5334, Train acc: 0.2934
Val loss: 1.6124, Val acc: 0.2667

Epoch 3/1000:
Train loss: 1.4928, Train acc: 0.3273
Val loss: 1.6459, Val acc: 0.2564

Epoch 4/1000:
Train loss: 1.4367, Train acc: 0.3681
Val loss: 1.6753, Val acc: 0.2642

Epoch 5/1000:
Train loss: 1.3834, Train acc: 0.4033
Val loss: 1.7804, Val acc: 0.2724

Epoch 6/1000:
Train loss: 1.3264, Train acc: 0.4387
Val loss: 1.7568, Val acc: 0.2633

Epoch 7/1000:
Train loss: 1.2570, Train acc: 0.4727
Val loss: 1.8168, Val acc: 0.2600

Epoch 8/1000:
Train loss: 1.2040, Train acc: 0.5094
Val loss: 2.1963, Val acc: 0.2484

Epoch 9/1000:
Train loss: 1.1465, Train acc: 0.5391
Val loss: 2.1145, Val acc: 0.2620

Epoch 10/1000:
Train loss: 1.0957, Train acc: 0.5572
Val loss: 2.1000, Val acc: 0.2640

Epoch 11/1000:
Train loss: 1.0430, Train 